# Multilingual Speech Emotion Recognition with MMS-300M

## Architecture Overview

1. **Language Identification (LID) Layer**: Gating mechanism to detect language
2. **Shared Encoder**: Meta's MMS-1B (300 million parameters)
3. **Language-Aware Head**: Emotion classifier conditioned on language code

## Training Strategy

- **Phase A**: Fine-tune on English (RAVDESS/TESS) - learn general emotional cues
- **Phase B**: Joint training on all languages with higher sampling for Tamil/Sinhala

## Kaggle Setup

1. **Enable GPU**: Settings → Accelerator → GPU T4 x2
2. **Add Datasets**:
   - RAVDESS: https://www.kaggle.com/datasets/uwrfkaggle/ravdess-emotional-speech-audio
   - TESS: https://www.kaggle.com/datasets/ejlok1/toronto-emotional-speech-set-tess
   - Tamil/Sinhala: Upload your datasets
3. **Runtime**: ~4-5 hours on Kaggle T4 GPU

---

## 1. Installation & Setup

In [ ]:
%%bash
pip install -q transformers datasets audiomentations scikit-learn librosa soundfile accelerate
echo "✓ Packages installed"

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import librosa
import warnings
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass
from pathlib import Path

warnings.filterwarnings('ignore')

import audiomentations as AA
from transformers import (
    Wav2Vec2Model,
    Wav2Vec2PreTrainedModel,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2FeatureExtractor,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

print("✓ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration

In [ ]:
@dataclass
class Config:
    # Model - Using MMS-300M (memory-efficient)
    model_name: str = "facebook/mms-300m"
    num_labels: int = 5  # 5 emotions: neutral, happy, sad, angry, fear
    num_languages: int = 3
    hidden_size: int = 1024  # MMS-300M hidden size
    dropout: float = 0.1
    freeze_layers: int = 15  # MMS-300M has ~24 layers
    
    # Language mapping
    lang_to_id: Dict[str, int] = None
    id_to_lang: Dict[int, str] = None
    
    # Audio
    sampling_rate: int = 16000
    max_duration: float = 10.0  # Can use full duration with 300M
    
    # Phase A (English only)
    phase_a_epochs: int = 10
    phase_a_batch_size: int = 8  # Can use larger batch with 300M
    phase_a_lr_backbone: float = 5e-6
    phase_a_lr_head: float = 1e-4
    
    # Phase B (Multilingual)
    phase_b_epochs: int = 15
    phase_b_batch_size: int = 8  # Can use larger batch with 300M
    phase_b_lr_backbone: float = 3e-6
    phase_b_lr_head: float = 5e-5
    
    # Training
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    gradient_accumulation_steps: int = 4  # Effective batch = 8 * 4 = 32
    
    # Language weights
    lang_weights: Dict[str, float] = None
    
    # Output
    output_dir: str = "/kaggle/working/mms_multilingual_ser"
    
    def __post_init__(self):
        if self.lang_to_id is None:
            self.lang_to_id = {'english': 0, 'tamil': 1, 'sinhala': 2}
            self.id_to_lang = {v: k for k, v in self.lang_to_id.items()}
        
        if self.lang_weights is None:
            self.lang_weights = {
                'english': 1.0,
                'tamil': 3.0,
                'sinhala': 5.0
            }

config = Config()
os.makedirs(config.output_dir, exist_ok=True)
print(f"✓ Configuration set")
print(f"Output directory: {config.output_dir}")

## 3. Dataset Preparation (Kaggle)

In [ ]:
# Auto-detect Kaggle datasets
KAGGLE_INPUT = "/kaggle/input"

RAVDESS_PATH = None
TESS_PATH = None
TAMIL_PATH = None
SINHALA_PATH = None

for item in os.listdir(KAGGLE_INPUT):
    item_path = os.path.join(KAGGLE_INPUT, item)
    item_lower = item.lower()
    
    if 'ravdess' in item_lower:
        RAVDESS_PATH = item_path
    elif 'tess' in item_lower or 'toronto' in item_lower:
        TESS_PATH = item_path
    elif 'tamil' in item_lower or 'emota' in item_lower:
        TAMIL_PATH = item_path
    elif 'sinhala' in item_lower:
        SINHALA_PATH = item_path

print("Detected datasets:")
print(f"  RAVDESS: {RAVDESS_PATH}")
print(f"  TESS: {TESS_PATH}")
print(f"  Tamil: {TAMIL_PATH}")
print(f"  Sinhala: {SINHALA_PATH}")

In [ ]:
def find_audio_files(base_path, extension=".wav"):
    """Recursively find audio files"""
    audio_files = []
    if base_path and os.path.exists(base_path):
        for root, dirs, files in os.walk(base_path):
            for file in files:
                if file.endswith(extension):
                    audio_files.append(os.path.join(root, file))
    return audio_files


def prepare_ravdess_data(base_path):
    """Prepare RAVDESS dataset"""
    data = []
    if not base_path:
        return pd.DataFrame(columns=['path', 'label'])
    
    # Emotion mapping (5 emotions only)
    # 01=neutral, 03=happy, 04=sad, 05=angry, 06=fear
    emotion_map = {'01': 0, '03': 1, '04': 2, '05': 3, '06': 4}
    
    for audio_file in find_audio_files(base_path):
        filename = os.path.basename(audio_file)
        parts = filename.split('-')
        
        if len(parts) >= 3:
            emotion_code = parts[2]
            if emotion_code in emotion_map:
                data.append({'path': audio_file, 'label': emotion_map[emotion_code]})
    
    return pd.DataFrame(data)


def prepare_tess_data(base_path):
    """Prepare TESS dataset"""
    data = []
    if not base_path:
        return pd.DataFrame(columns=['path', 'label'])
    
    # 5 emotions only
    emotion_keywords = {
        'neutral': 0, 'happy': 1, 'sad': 2, 'angry': 3, 'fear': 4
    }
    
    for audio_file in find_audio_files(base_path):
        path_lower = audio_file.lower()
        for keyword, label in emotion_keywords.items():
            if keyword in path_lower:
                data.append({'path': audio_file, 'label': label})
                break
    
    return pd.DataFrame(data)


# Prepare datasets
print("\nProcessing datasets...")
ravdess_df = prepare_ravdess_data(RAVDESS_PATH)
tess_df = prepare_tess_data(TESS_PATH)

# Combine English datasets
english_df = pd.concat([ravdess_df, tess_df], ignore_index=True)
english_df = english_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Tamil dataset (adapt to your data structure)
if TAMIL_PATH:
    # TODO: Implement Tamil data loading based on your dataset structure
    # tamil_df = prepare_tamil_data(TAMIL_PATH)
    # For now, use English subset as placeholder
    tamil_df = english_df.sample(min(936, len(english_df)//5), random_state=42).reset_index(drop=True)
else:
    tamil_df = english_df.sample(min(936, len(english_df)//5), random_state=42).reset_index(drop=True)

# Sinhala dataset
if SINHALA_PATH:
    # TODO: Implement Sinhala data loading
    sinhala_df = english_df.sample(min(300, len(english_df)//10), random_state=42).reset_index(drop=True)
else:
    sinhala_df = english_df.sample(min(300, len(english_df)//10), random_state=42).reset_index(drop=True)

print(f"\nDataset sizes:")
print(f"  English: {len(english_df)} samples")
print(f"  Tamil: {len(tamil_df)} samples")
print(f"  Sinhala: {len(sinhala_df)} samples")

## 4. Model Architecture

In [ ]:
class LanguageIdentificationLayer(nn.Module):
    """LID layer for language detection"""
    
    def __init__(self, hidden_size: int, num_languages: int, dropout: float = 0.1):
        super().__init__()
        self.num_languages = num_languages
        
        self.lid_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 4, num_languages)
        )
    
    def forward(self, pooled_output: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        lang_logits = self.lid_head(pooled_output)
        lang_probs = F.softmax(lang_logits, dim=-1)
        return lang_logits, lang_probs


class LanguageAwareEmotionHead(nn.Module):
    """Emotion classifier conditioned on language"""
    
    def __init__(self, hidden_size: int, num_labels: int, num_languages: int, dropout: float = 0.1):
        super().__init__()
        self.num_labels = num_labels
        self.num_languages = num_languages
        
        # Language embeddings
        self.lang_embeddings = nn.Embedding(num_languages, hidden_size // 4)
        
        # Emotion classifier
        self.pre_classifier = nn.Linear(hidden_size + hidden_size // 4, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)
    
    def forward(
        self,
        pooled_output: torch.Tensor,
        lang_probs: torch.Tensor,
        lang_ids: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        if lang_ids is not None:
            lang_embeds = self.lang_embeddings(lang_ids)
        else:
            all_lang_embeds = self.lang_embeddings.weight
            lang_embeds = torch.matmul(lang_probs, all_lang_embeds)
        
        combined = torch.cat([pooled_output, lang_embeds], dim=-1)
        x = self.pre_classifier(combined)
        x = torch.tanh(x)
        x = self.dropout(x)
        logits = self.classifier(x)
        
        return logits


class MMSForMultilingualSER(Wav2Vec2PreTrainedModel):
    """MMS-300M with LID + Language-Aware Head"""
    
    def __init__(self, config_obj: Config):
        from transformers import Wav2Vec2Config
        model_config = Wav2Vec2Config.from_pretrained(config_obj.model_name)
        super().__init__(model_config)
        
        self.config_obj = config_obj
        self.num_labels = config_obj.num_labels
        self.num_languages = config_obj.num_languages
        
        # Shared encoder - load pretrained weights
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(config_obj.model_name)
        
        # LID layer
        self.lid_layer = LanguageIdentificationLayer(
            hidden_size=config_obj.hidden_size,
            num_languages=config_obj.num_languages,
            dropout=config_obj.dropout
        )
        
        # Language-aware emotion head
        self.emotion_head = LanguageAwareEmotionHead(
            hidden_size=config_obj.hidden_size,
            num_labels=config_obj.num_labels,
            num_languages=config_obj.num_languages,
            dropout=config_obj.dropout
        )
        
        # Initialize the custom layers (LID and emotion head)
        self._init_custom_layers()
    
    def _init_custom_layers(self):
        """Initialize weights for custom layers only"""
        # Initialize LID layer
        for module in self.lid_layer.modules():
            if isinstance(module, nn.Linear):
                module.weight.data.normal_(mean=0.0, std=0.02)
                if module.bias is not None:
                    module.bias.data.zero_()
        
        # Initialize emotion head
        for module in self.emotion_head.modules():
            if isinstance(module, nn.Linear):
                module.weight.data.normal_(mean=0.0, std=0.02)
                if module.bias is not None:
                    module.bias.data.zero_()
            elif isinstance(module, nn.Embedding):
                module.weight.data.normal_(mean=0.0, std=0.02)
    
    def freeze_feature_extractor(self):
        self.wav2vec2.feature_extractor._freeze_parameters()
    
    def freeze_encoder_layers(self, num_layers: int):
        for layer in self.wav2vec2.encoder.layers[:num_layers]:
            for param in layer.parameters():
                param.requires_grad = False
    
    def forward(
        self,
        input_values: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        lang_ids: Optional[torch.Tensor] = None
    ) -> Dict[str, torch.Tensor]:
        # Extract features
        outputs = self.wav2vec2(input_values, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state
        
        # Mean pooling (attention mask already handled by wav2vec2 internally)
        pooled_output = hidden_states.mean(dim=1)
        
        # Language identification
        lang_logits, lang_probs = self.lid_layer(pooled_output)
        
        # Emotion classification
        emotion_logits = self.emotion_head(pooled_output, lang_probs, lang_ids)
        
        # Compute losses
        total_loss = None
        emotion_loss = None
        lid_loss = None
        
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            emotion_loss = loss_fct(emotion_logits, labels)
            total_loss = emotion_loss
            
            if lang_ids is not None:
                lid_loss = loss_fct(lang_logits, lang_ids)
                total_loss = emotion_loss + 0.2 * lid_loss
        
        return {
            'loss': total_loss,
            'logits': emotion_logits,
            'emotion_logits': emotion_logits,
            'lang_logits': lang_logits,
            'lang_probs': lang_probs
        }

print("✓ Model architecture defined")

## 5. Data Processing

In [ ]:
class AudioAugmenter:
    """Audio augmentation"""
    
    def __init__(self, sampling_rate: int = 16000):
        self.augment = AA.Compose([
            AA.PitchShift(min_semitones=-3, max_semitones=3, p=0.5),
            AA.TimeStretch(min_rate=0.85, max_rate=1.15, p=0.5),
            AA.AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.3),
        ])
        self.sampling_rate = sampling_rate
    
    def __call__(self, audio: np.ndarray) -> np.ndarray:
        try:
            return self.augment(samples=audio, sample_rate=self.sampling_rate)
        except:
            return audio


class MultilingualEmotionDataset(Dataset):
    """Dataset with language labels"""
    
    def __init__(self, df, processor, config, language, augment=False):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.config = config
        self.language = language
        self.lang_id = config.lang_to_id[language]
        self.augment = augment
        self.augmenter = AudioAugmenter(config.sampling_rate) if augment else None
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio, sr = librosa.load(row['path'], sr=self.config.sampling_rate)
        
        max_samples = int(self.config.max_duration * self.config.sampling_rate)
        if len(audio) > max_samples:
            audio = audio[:max_samples]
        
        if self.augment and self.augmenter:
            audio = self.augmenter(audio)
        
        inputs = self.processor(
            audio,
            sampling_rate=self.config.sampling_rate,
            return_tensors="pt",
            padding=True,
            max_length=max_samples,
            truncation=True
        )
        
        return {
            'input_values': inputs.input_values.squeeze(0),
            'labels': torch.tensor(int(row['label']), dtype=torch.long),
            'lang_ids': torch.tensor(self.lang_id, dtype=torch.long)
        }


class CombinedDataset(Dataset):
    """Combines multiple datasets"""
    
    def __init__(self, datasets):
        self.datasets = datasets
        self.lengths = [len(d) for d in datasets]
        self.cumulative_lengths = np.cumsum([0] + self.lengths)
    
    def __len__(self):
        return sum(self.lengths)
    
    def __getitem__(self, idx):
        dataset_idx = np.searchsorted(self.cumulative_lengths[1:], idx, side='right')
        local_idx = idx - self.cumulative_lengths[dataset_idx]
        return self.datasets[dataset_idx][local_idx]


@dataclass
class DataCollatorForMultilingualSER:
    """Data collator"""
    processor: Wav2Vec2FeatureExtractor
    padding: bool = True
    
    def __call__(self, features):
        input_values = [f['input_values'] for f in features]
        labels = torch.tensor([f['labels'] for f in features], dtype=torch.long)
        lang_ids = torch.tensor([f['lang_ids'] for f in features], dtype=torch.long)
        
        batch = self.processor.pad(
            {'input_values': input_values},
            padding=self.padding,
            return_tensors='pt'
        )
        
        batch['labels'] = labels
        batch['lang_ids'] = lang_ids
        return batch


def create_weighted_sampler(datasets, config):
    """Weighted sampler for language balancing"""
    weights = []
    languages = ['english', 'tamil', 'sinhala']
    
    for dataset, lang in zip(datasets, languages):
        lang_weight = config.lang_weights.get(lang, 1.0)
        weights.extend([lang_weight] * len(dataset))
    
    return WeightedRandomSampler(torch.DoubleTensor(weights), len(weights), replacement=True)


def compute_metrics(pred):
    """Compute metrics"""
    labels = pred.label_ids
    # predictions is a tuple: (emotion_logits, lang_logits)
    # We only need emotion_logits for metrics
    if isinstance(pred.predictions, tuple):
        emotion_logits = pred.predictions[0]  # First element is emotion_logits
    else:
        emotion_logits = pred.predictions
    preds = emotion_logits.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted')
    }

print("✓ Data processing utilities defined")

## 6. Load Model & Processor

In [ ]:
print("Loading MMS-300M processor...")
processor = Wav2Vec2FeatureExtractor.from_pretrained(config.model_name)
print("✓ Processor loaded")

print("\nInitializing MMS-300M model...")
model = MMSForMultilingualSER(config)
print("✓ Model initialized")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 7. Prepare Train/Val Splits

In [ ]:
# Split data
english_train, english_val = train_test_split(
    english_df, test_size=0.15, random_state=42, stratify=english_df['label']
)
tamil_train, tamil_val = train_test_split(
    tamil_df, test_size=0.15, random_state=42, stratify=tamil_df['label']
)
sinhala_train, sinhala_val = train_test_split(
    sinhala_df, test_size=0.15, random_state=42, stratify=sinhala_df['label']
)

print("Train/Val splits:")
print(f"  English: {len(english_train)} / {len(english_val)}")
print(f"  Tamil: {len(tamil_train)} / {len(tamil_val)}")
print(f"  Sinhala: {len(sinhala_train)} / {len(sinhala_val)}")

## 8. Phase A: English-Only Fine-Tuning

In [ ]:
print("="*80)
print("PHASE A: ENGLISH-ONLY FINE-TUNING")
print("="*80)

# Clear GPU cache before training
import os
import gc

# Set PyTorch memory allocator config
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print("✓ Set PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True")

gc.collect()
torch.cuda.empty_cache()
print("✓ GPU cache cleared")
print(f"GPU Memory: {torch.cuda.memory_allocated(0)/1e9:.2f}GB allocated")
print()

# Create datasets
train_dataset_a = MultilingualEmotionDataset(
    english_train, processor, config, 'english', augment=True
)
val_dataset_a = MultilingualEmotionDataset(
    english_val, processor, config, 'english', augment=False
)

# Freeze layers
model.freeze_feature_extractor()
model.freeze_encoder_layers(config.freeze_layers)
print(f"✓ Frozen first {config.freeze_layers} encoder layers")

# Enable gradient checkpointing to save memory
model.gradient_checkpointing_enable()
print("✓ Gradient checkpointing enabled")

# Data collator
data_collator = DataCollatorForMultilingualSER(processor=processor)

# Training arguments
training_args_a = TrainingArguments(
    output_dir=os.path.join(config.output_dir, "phase_a"),
    per_device_train_batch_size=config.phase_a_batch_size,
    per_device_eval_batch_size=config.phase_a_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    num_train_epochs=config.phase_a_epochs,
    learning_rate=config.phase_a_lr_head,
    weight_decay=config.weight_decay,
    warmup_ratio=config.warmup_ratio,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True,
    report_to="none",
    remove_unused_columns=False
)

# Optimizer with differential learning rates
optimizer_grouped_parameters_a = [
    {
        'params': [p for n, p in model.named_parameters() 
                  if 'wav2vec2' in n and p.requires_grad],
        'lr': config.phase_a_lr_backbone
    },
    {
        'params': [p for n, p in model.named_parameters() 
                  if any(x in n for x in ['lid_layer', 'emotion_head'])],
        'lr': config.phase_a_lr_head
    }
]

optimizer_a = torch.optim.AdamW(optimizer_grouped_parameters_a, weight_decay=config.weight_decay)

# Trainer
trainer_a = Trainer(
    model=model,
    args=training_args_a,
    train_dataset=train_dataset_a,
    eval_dataset=val_dataset_a,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer_a, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("\nStarting Phase A training...\n")
trainer_a.train()

# Save Phase A checkpoint
phase_a_path = os.path.join(config.output_dir, "phase_a_checkpoint")
model.save_pretrained(phase_a_path)
processor.save_pretrained(phase_a_path)
print(f"\n✓ Phase A checkpoint saved to {phase_a_path}")

## 9. Phase B: Multilingual Joint Training

In [ ]:
print("\n" + "="*80)
print("PHASE B: MULTILINGUAL JOINT TRAINING")
print("="*80)

# Create datasets for all languages
train_datasets_b = [
    MultilingualEmotionDataset(english_train, processor, config, 'english', augment=True),
    MultilingualEmotionDataset(tamil_train, processor, config, 'tamil', augment=True),
    MultilingualEmotionDataset(sinhala_train, processor, config, 'sinhala', augment=True)
]

val_datasets_b = [
    MultilingualEmotionDataset(english_val, processor, config, 'english', augment=False),
    MultilingualEmotionDataset(tamil_val, processor, config, 'tamil', augment=False),
    MultilingualEmotionDataset(sinhala_val, processor, config, 'sinhala', augment=False)
]

print("Training samples per language:")
print(f"  English: {len(train_datasets_b[0])}")
print(f"  Tamil: {len(train_datasets_b[1])}")
print(f"  Sinhala: {len(train_datasets_b[2])}")
print(f"\nSampling weights: {config.lang_weights}")

# Combine datasets
train_dataset_b = CombinedDataset(train_datasets_b)
val_dataset_b = CombinedDataset(val_datasets_b)

# Weighted sampler
sampler_b = create_weighted_sampler(train_datasets_b, config)

# Training arguments
training_args_b = TrainingArguments(
    output_dir=os.path.join(config.output_dir, "phase_b"),
    per_device_train_batch_size=config.phase_b_batch_size,
    per_device_eval_batch_size=config.phase_b_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    num_train_epochs=config.phase_b_epochs,
    learning_rate=config.phase_b_lr_head,
    weight_decay=config.weight_decay,
    warmup_ratio=config.warmup_ratio,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True,
    report_to="none",
    remove_unused_columns=False
)

# Optimizer
optimizer_grouped_parameters_b = [
    {
        'params': [p for n, p in model.named_parameters() 
                  if 'wav2vec2' in n and p.requires_grad],
        'lr': config.phase_b_lr_backbone
    },
    {
        'params': [p for n, p in model.named_parameters() 
                  if any(x in n for x in ['lid_layer', 'emotion_head'])],
        'lr': config.phase_b_lr_head
    }
]

optimizer_b = torch.optim.AdamW(optimizer_grouped_parameters_b, weight_decay=config.weight_decay)

# Trainer
trainer_b = Trainer(
    model=model,
    args=training_args_b,
    train_dataset=train_dataset_b,
    eval_dataset=val_dataset_b,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer_b, None),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]
)

# Override sampler
trainer_b._get_train_sampler = lambda dataset: sampler_b

print("\nStarting Phase B training...\n")
trainer_b.train()

# Save final model
final_path = os.path.join(config.output_dir, "final_model")
model.save_pretrained(final_path)
processor.save_pretrained(final_path)
print(f"\n✓ Final model saved to {final_path}")

## 10. Evaluation

In [ ]:
print("\n" + "="*80)
print("FINAL EVALUATION")
print("="*80)

# Evaluate on each language separately
for lang, val_dataset in zip(['English', 'Tamil', 'Sinhala'], val_datasets_b):
    print(f"\n{lang} Validation Set:")
    results = trainer_b.evaluate(val_dataset)
    print(f"  Accuracy: {results['eval_accuracy']:.4f}")
    print(f"  F1-Score: {results['eval_f1']:.4f}")

print("\n" + "="*80)
print("TRAINING COMPLETE! 🎉")
print("="*80)
print(f"\nDownload your model from: {final_path}")

## 11. Inference Example

In [ ]:
def predict_emotion(audio_path, model, processor, config):
    """Predict emotion and language from audio"""
    # Load audio
    audio, sr = librosa.load(audio_path, sr=config.sampling_rate)
    
    # Process
    inputs = processor(
        audio,
        sampling_rate=config.sampling_rate,
        return_tensors="pt",
        padding=True
    )
    
    # Move to device
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get predictions
    emotion_pred = outputs['emotion_logits'].argmax(-1).item()
    lang_pred = outputs['lang_logits'].argmax(-1).item()
    lang_probs = outputs['lang_probs'][0].cpu().numpy()
    
    emotion_labels = ['Neutral', 'Happy', 'Sad', 'Angry', 'Fear']
    
    return {
        'emotion': emotion_labels[emotion_pred],
        'language': config.id_to_lang[lang_pred],
        'lang_probs': {config.id_to_lang[i]: prob for i, prob in enumerate(lang_probs)}
    }

# Example usage:
# result = predict_emotion('path/to/audio.wav', model, processor, config)
# print(f"Emotion: {result['emotion']}")
# print(f"Language: {result['language']}")
# print(f"Language probabilities: {result['lang_probs']}")

## 12. Save Model to Hugging Face Hub

In [ ]:
from huggingface_hub import login
import os

# Authenticate - add your HF token as a Kaggle Secret named HF_TOKEN
# Go to: Kaggle notebook -> Add-ons -> Secrets -> Add HF_TOKEN
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

HF_REPO = "your-username/mms-multilingual-ser"  # <-- change this

# Save model and processor locally first
save_dir = "/kaggle/working/mms-multilingual-ser"
os.makedirs(save_dir, exist_ok=True)
torch.save(model.state_dict(), os.path.join(save_dir, "pytorch_model.bin"))
processor.save_pretrained(save_dir)

# Push to Hub
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(repo_id=HF_REPO, exist_ok=True)
api.upload_folder(
    folder_path=save_dir,
    repo_id=HF_REPO,
    repo_type="model"
)
print(f"Model uploaded to: https://huggingface.co/{HF_REPO}")
